# 약물 처방(Medication) 기반 회복 기간 예측 모델 (2단계: 가상 데이터셋)

**배경:** 1단계(`01_athlete_recovery_analysis.ipynb`)에서 실제 데이터셋의 변수 간 상관관계가 거의 0에 가까워, 모델의 R²가 음수로 나타났다. 이에 본 노트북에서는 약리학적 가정을 반영한 가상(synthetic) 데이터셋을 생성하여, 동일한 파이프라인으로 약물(NSAIDs/스테로이드) 처방이 회복 기간에 미치는 영향을 검증한다.

**주의:** 본 데이터셋은 실제 임상 데이터가 아닌, 약리학적 가정을 기반으로 생성한 시뮬레이션 데이터이다.

## 1. 라이브러리 임포트 및 시드 설정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)
N = 1000

## 2. 가상 데이터셋 생성

### 설계 가정
- 부상 심각도가 높을수록 스테로이드(Steroid) 처방 비율 증가
- 부상 종류·심각도별 기본 회복 기간 차등 설정 (ACL Tear > Sprained Ankle > Muscle Strain)
- NSAIDs(-8일), Steroid(-14일) 처방 시 회복 기간 단축 (급성 염증 완화 효과 가정)
- 물리치료(Physiotherapy, -6일) 및 충분한 수면이 회복 기간 단축에 긍정적 영향
- 현실적 분석 환경 재현을 위해 기본값의 50% 수준 정규분포 노이즈 추가

In [ ]:
# 2-1. 기본 변수 생성
injury_types = np.random.choice(['Sprained Ankle', 'Muscle Strain', 'ACL Tear'], N, p=[0.35, 0.4, 0.25])
injury_severity = np.random.choice(['Mild', 'Moderate', 'Severe'], N, p=[0.35, 0.4, 0.25])
age = np.random.randint(18, 40, N)
sex = np.random.choice(['Male', 'Female'], N)

# 2-2. 약물 처방 (부상 심각도에 따라 처방 확률 차등)
medication = []
for sev in injury_severity:
    if sev == 'Mild':
        medication.append(np.random.choice(['None', 'NSAIDs', 'Steroid'], p=[0.4, 0.55, 0.05]))
    elif sev == 'Moderate':
        medication.append(np.random.choice(['None', 'NSAIDs', 'Steroid'], p=[0.15, 0.6, 0.25]))
    else:  # Severe
        medication.append(np.random.choice(['None', 'NSAIDs', 'Steroid'], p=[0.05, 0.45, 0.5]))
medication = np.array(medication)

# 2-3. 생활습관/치료법
sleep_hours = np.random.normal(7, 1.2, N).clip(3, 10)
training_intensity = np.random.choice(['Low', 'Medium', 'High'], N)
therapy_type = np.random.choice(['Stretching', 'Rest and Ice', 'Physiotherapy'], N)

print('변수 생성 완료')

In [ ]:
# 2-4. Recovery_Time 생성 (약리학적 가정 + 노이즈)
base_days = {
    ('Sprained Ankle', 'Mild'): 14, ('Sprained Ankle', 'Moderate'): 24, ('Sprained Ankle', 'Severe'): 35,
    ('Muscle Strain', 'Mild'): 12, ('Muscle Strain', 'Moderate'): 22, ('Muscle Strain', 'Severe'): 33,
    ('ACL Tear', 'Mild'): 35, ('ACL Tear', 'Moderate'): 50, ('ACL Tear', 'Severe'): 70,
}

medication_effect = {'None': 0, 'NSAIDs': -8, 'Steroid': -14}
therapy_effect = {'Stretching': 0, 'Rest and Ice': -2, 'Physiotherapy': -6}

recovery_time = []
for i in range(N):
    base = base_days[(injury_types[i], injury_severity[i])]
    med_eff = medication_effect[medication[i]]
    therapy_eff = therapy_effect[therapy_type[i]]
    sleep_eff = -1.5 * (sleep_hours[i] - 7)
    noise = np.random.normal(0, base * 0.5)
    rt = base + med_eff + therapy_eff + sleep_eff + noise
    recovery_time.append(max(3, round(rt)))

df = pd.DataFrame({
    'Injury_Type': injury_types,
    'Injury_Severity': injury_severity,
    'Age': age,
    'Sex': sex,
    'Medication': medication,
    'Sleep_Hours': np.round(sleep_hours, 1),
    'Training_Intensity': training_intensity,
    'Recovery_Therapy_Type': therapy_type,
    'Recovery_Time': recovery_time
})

df.to_csv('synthetic_drug_recovery_dataset.csv', index=False)
df.head()

## 3. 탐색적 데이터 분석 (EDA)

In [ ]:
print('[처방 약물별 평균 회복 기간]')
print(df.groupby('Medication')['Recovery_Time'].mean())

plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x='Medication', y='Recovery_Time', order=['None', 'NSAIDs', 'Steroid'])
plt.title('Recovery Time by Medication Type')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x='Injury_Severity', y='Recovery_Time', order=['Mild', 'Moderate', 'Severe'])
plt.title('Recovery Time by Injury Severity')
plt.tight_layout()
plt.show()

**해석:** 약물 처방군별 평균 회복 기간은 None(무처방) < NSAIDs < Steroid 순으로 나타났다. 이는 설계 가정(스테로이드는 부상이 심한 경우에 더 많이 처방되며, 약물 자체의 단축 효과보다 기저 부상 심각도의 영향이 더 크게 반영됨)과 일치하는 패턴이다.

## 4. 데이터 전처리

In [ ]:
ordinal_map = {'Mild': 0, 'Moderate': 1, 'Severe': 2}
df['Injury_Severity_Enc'] = df['Injury_Severity'].map(ordinal_map)

intensity_map = {'Low': 0, 'Medium': 1, 'High': 2}
df['Training_Intensity_Enc'] = df['Training_Intensity'].map(intensity_map)

df_model = pd.get_dummies(
    df.drop(columns=['Injury_Severity', 'Training_Intensity']),
    columns=['Injury_Type', 'Sex', 'Medication', 'Recovery_Therapy_Type'],
    drop_first=True
)

print('전처리 후 컬럼:', df_model.columns.tolist())
df_model.head()

## 5. Feature/Target 분리, 데이터 분할 및 모델링

In [ ]:
y = df_model['Recovery_Time']
X = df_model.drop(columns=['Recovery_Time'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

results = {}

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
results['Linear Regression'] = lr.predict(X_test_scaled)

rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
results['Random Forest'] = rf.predict(X_test)

print('모델 학습 완료')

## 6. 모델 평가

In [ ]:
eval_table = []
for name, y_pred in results.items():
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    eval_table.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    print(f'{name:18s} | MAE: {mae:.3f} | RMSE: {rmse:.3f} | R2: {r2:.3f}')

pd.DataFrame(eval_table)

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, results['Random Forest'], alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Recovery Time')
plt.ylabel('Predicted Recovery Time')
plt.title('Random Forest: Actual vs Predicted (Synthetic Data)')
plt.tight_layout()
plt.show()

**해석:** R² 0.37 ~ 0.41 수준으로, 1단계(R² ≈ -0.05) 대비 유의미한 예측력을 확보하였다. 이는 변수 간 신호가 존재하는 데이터에서는 동일한 모델링 파이프라인이 정상적으로 작동함을 보여준다.

## 7. Feature Importance 분석

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances.head(10))

plt.figure(figsize=(8, 6))
importances.head(10).sort_values().plot(kind='barh')
plt.title('Top 10 Feature Importances (Synthetic Data)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

**해석:** 부상 종류(Injury_Type) 및 심각도(Injury_Severity_Enc), 수면 시간(Sleep_Hours), 나이(Age)가 회복 기간에 가장 큰 영향을 미치는 요인으로 나타났다. 약물 처방 변수(Medication_Steroid)도 상위 변수에 포함되어, 약물 처방이 회복 기간 예측에 기여할 수 있음을 보여준다.

## 8. 결론 (2단계)

- 약리학적 가정을 반영한 가상 데이터셋을 통해, 1단계와 동일한 파이프라인으로 R² 약 0.4 수준의 예측력을 확보하였다.
- 부상 종류/심각도, 수면, 약물 처방(특히 Steroid)이 회복 기간에 영향을 미치는 주요 변수로 확인되었다.
- **한계:** 본 데이터셋은 실제 임상 데이터가 아닌 시뮬레이션 데이터이다. 향후 연구에서는 실제 NSAIDs/스테로이드 처방 데이터(예: 건강보험공단 의약품처방정보)를 확보하여 본 파이프라인을 재적용할 필요가 있다.
- **추가 한계:** 1단계의 분류 모델(Recovery_Success)에서 나타난 클래스 불균형 문제는 SMOTE 등 오버샘플링 기법으로 개선 가능하다.